In [11]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [12]:
@tool
def convert_temperature(value: float, direction: str) -> str:
    """Convert temperature between Celsius and Fahrenheit.
    direction must be 'C_to_F' or 'F_to_C'.
    """

    direction = direction.strip().upper()

    if direction == "C_TO_F":
        result = (value * 9 / 5) + 32
        return f"{value}°C = {result:.2f}°F"

    elif direction == "F_TO_C":
        result = (value - 32) * 5 / 9
        return f"{value}°F = {result:.2f}°C"

    else:
        raise ValueError(
            "Invalid direction. Use 'C_to_F' or 'F_to_C'."
        )


print("Temperature conversion tool created.")

print(
    convert_temperature.invoke({
        "value": 100,
        "direction": "C_to_F"
    })
)

Temperature conversion tool created.
100.0°C = 212.00°F


In [13]:
@tool
def convert_distance(value: float, direction: str) -> str:
    """Convert distance between kilometers and miles.
    direction must be 'km_to_miles' or 'miles_to_km'.
    """

    direction = direction.strip().lower()

    if direction == "km_to_miles":
        result = value * 0.621371
        return f"{value} km = {result:.2f} miles"

    elif direction == "miles_to_km":
        result = value / 0.621371
        return f"{value} miles = {result:.2f} km"

    else:
        raise ValueError(
            "Invalid direction. Use 'km_to_miles' or 'miles_to_km'."
        )


print("Distance conversion tool created.")

print(
    convert_distance.invoke({
        "value": 100,
        "direction": "km_to_miles"
    })
)

Distance conversion tool created.
100.0 km = 62.14 miles


In [14]:
tools = [
    convert_temperature,
    convert_distance
]

llm_with_tools = llm.bind_tools(tools)

print("Both tools bound to the model.")

for item in tools:
    print("-", item.name)

Both tools bound to the model.
- convert_temperature
- convert_distance


In [15]:
user_query = "Convert 100 km to miles."

messages = [
    HumanMessage(content=user_query)
]

response = llm_with_tools.invoke(messages)

print("User query:")
print(user_query)

print("\nModel-suggested tool calls:")

for call in response.tool_calls:
    print("Tool:", call["name"])
    print("Arguments:", call["args"])

if not response.tool_calls:
    raise RuntimeError(
        "The model did not suggest a tool call. Run this cell again."
    )

User query:
Convert 100 km to miles.

Model-suggested tool calls:
Tool: convert_distance
Arguments: {'direction': 'km_to_miles', 'value': 100}


In [16]:
tool_map = {
    "convert_temperature": convert_temperature,
    "convert_distance": convert_distance
}

messages.append(response)
tool_results = []

for tool_call in response.tool_calls:
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    if tool_name not in tool_map:
        raise ValueError(f"Unknown tool requested: {tool_name}")

    result = tool_map[tool_name].invoke(tool_args)
    tool_results.append((tool_name, result))

    print("\nExecuted tool:", tool_name)
    print("Arguments:", tool_args)
    print("Result:", result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    )

print("\nManual tool execution completed.")


Executed tool: convert_distance
Arguments: {'direction': 'km_to_miles', 'value': 100}
Result: 100.0 km = 62.14 miles

Manual tool execution completed.


In [17]:
final_response = llm.invoke(messages)

print("Final answer:")
print(final_response.content)

Final answer:
100 km is approximately **62.14 miles**.


In [18]:
assert response.tool_calls, "No tool call was suggested."

selected_tools = [
    call["name"] for call in response.tool_calls
]

assert "convert_distance" in selected_tools, (
    "The distance conversion tool was not selected."
)

assert "convert_temperature" not in selected_tools, (
    "The temperature tool should not be needed for this query."
)

assert any(
    name == "convert_distance" and "62.14 miles" in result
    for name, result in tool_results
), "Expected distance conversion result was not returned."

assert final_response.content.strip(), "Final answer is empty."

print("Validation passed.")
print("The correct tool was selected.")
print("The temperature tool was not called.")
print("The distance conversion result was returned.")
print("The final answer was generated.")

Validation passed.
The correct tool was selected.
The temperature tool was not called.
The distance conversion result was returned.
The final answer was generated.


In [19]:
temperature_test = convert_temperature.invoke({
    "value": 32,
    "direction": "F_to_C"
})

distance_test = convert_distance.invoke({
    "value": 1,
    "direction": "miles_to_km"
})

print("Temperature test:", temperature_test)
print("Distance test:", distance_test)

assert "0.00°C" in temperature_test
assert "1.61 km" in distance_test

print("Reverse conversion tests passed.")

Temperature test: 32.0°F = 0.00°C
Distance test: 1.0 miles = 1.61 km
Reverse conversion tests passed.


In [20]:
print("Lab 6 - Exercise 3 completed successfully.")
print("Two @tool functions were created.")
print("Both tools were bound to the model in one call.")
print("The model selected the relevant tool.")
print("The suggested tool call was executed manually.")
print("The final answer and reverse conversions were tested.")

Lab 6 - Exercise 3 completed successfully.
Two @tool functions were created.
Both tools were bound to the model in one call.
The model selected the relevant tool.
The suggested tool call was executed manually.
The final answer and reverse conversions were tested.
